<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08-heatsink-cooling/Ex08_heatsink_cooling_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08 — Heatsink Cooling of a Power Module

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.1 · Stationary Heat and L8.2 · Transient Heat**

A power module sits on an aluminium heat sink and dissipates 150 W; the heat
spreads through the base plate and leaves through ten fins into forced air.
This notebook asks two questions of the same heat sink, one per lecture.
**Part 1 (L8.1)**: in steady operation, how hot does the module's footprint
get, and how far above what a hand calculation says? **Part 2 (L8.2)**: when
the module is switched on for twenty seconds and off, how hot does it get, how
long does the sink take to recover - and can one thermocouple tell us the
fins' heat transfer coefficient, the one number nobody knows to better than
twenty per cent? Both parts have an **exact solution**, so finite differences
(FDM) and a physics-informed neural network (PINN) are scored against the
truth, and their times are compared on what runs online.

**One notebook, one session.** Part 2 continues where Part 1 stops: the
libraries, the heat sink and Part 1's results stay in memory, and nothing is
loaded again.

| | what you do | TODO |
|---|---|---|
| **Part 1 · 1 – 3** | the problem, its data and its physics: the fins, the base plate, the exact solution | — |
| **4** | FDM on a grid, refined until an agreed 0.01 K | — |
| **5 – 6** | a PINN with the hand calculation built in, the comparison, what it says | TODO 1 and 2 |
| **Part 2 · 7 – 9** | the pulse: storage, two time scales, the exact transient | — |
| **10** | Crank–Nicolson, refined until an agreed 0.05 K | — |
| **11** | a PINN in time with the lumped response built in, and the comparison | TODO 3 and 4 |
| **12 – 13** | the inverse problem: $h_{\mathrm{eff}}$ from one thermocouple; what it says | — |
| **14 – 15** | your report, and the mini project proposals | your answers |

About twenty-five minutes, most of it the three trainings, on a CPU or a GPU
alike: networks this small do not fill a GPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'heatsink.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "heatsink"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "heatsink.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import heatsink as hs
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex08_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def online(fn, reps=5):                                    # the same, with the GPU made to finish before the clock stops
    def run():
        fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return middle(run, reps)

---

# Part 1 · Stationary heat (L8.1)

---

## 1 · The problem

An extruded aluminium heat sink: a base plate 100 mm wide and 10 mm thick with
ten straight fins, 2 mm thick and 40 mm long, at a 10 mm pitch. Forced air at
40 °C flows along the fins. A power module 30 mm wide sits in the middle of
the base plate and dissipates 150 W per 100 mm of depth, evenly over its
footprint. The ends of the base plate are insulated, and so is the top face
outside the module. What is the temperature at every point of the base plate,
and how hot is the hottest point under the module?

## 2 · The data

| | |
|---|---|
| base plate | 100 × 10 mm, aluminium alloy 6061, $k$ = 167 W/(m·K) |
| fins | 10 straight fins, 2 × 40 mm, pitch 10 mm |
| air | forced, 40 °C, $h$ = 40 W/(m²K) on the fins and the bare base — *typical value* |
| module | 150 W on a 30 mm wide footprint, centred; $q''$ = 50 kW/m² — *typical values* |
| elsewhere | the ends of the base and the top face outside the module are insulated |

The numbers are of the right order for a small drive and were not measured on
any product; every result below depends on them.

## 3 · The physics

**The fins enter exactly.** A straight fin is the first problem of L8.1 with
an exact solution: conduction along it, convection from both faces, and the
textbook efficiency

$$\eta = \frac{\tanh(mL_c)}{mL_c}, \qquad m = \sqrt{\frac{2h}{k\,t_f}}, \qquad L_c = L_f + \frac{t_f}{2}$$

(Incropera ch. 3). Per pitch $p$ of base the fin and the bare base beside it
take $h\,(\eta\, 2L_c + p - t_f)$ watts per kelvin and metre of depth, so the
finned face acts on the base plate as **one effective coefficient**

$$h_{\mathrm{eff}} = h\,\frac{\eta\,2L_c + p - t_f}{p}$$

That is how a heat sink is sized by hand, and it is what turns the base plate
into a rectangle. (How good the collapse is, with the fins resolved, is mini
project MP8.1A.)

**The base plate** is steady conduction with all three kinds of wall of
L8.1's *Thermal Boundary Conditions*: a **flux** on the module's footprint,
**convection** on the fin side, and **insulated** ends.

$$k\,(T_{xx} + T_{zz}) = 0, \qquad
k\,T_z = q''(x)\ \text{at}\ z = t_b, \qquad
k\,T_z = h_{\mathrm{eff}}\,(T - T_\infty)\ \text{at}\ z = 0, \qquad
T_x = 0\ \text{at}\ x = 0, W$$

No wall holds a temperature. The level is fixed by the convection: the heat
that enters must leave, and leaves only at a rate $h_{\mathrm{eff}}$ times the
rise. **The hand calculation** uses exactly that: spread the 150 W over the
whole base, $q_{\mathrm{mean}}$ = 15 kW/m², and the fin side sits
$q_{\mathrm{mean}}/h_{\mathrm{eff}}$ above the air, the module side another
$q_{\mathrm{mean}}\,t_b/k$ higher. It is one dimension, and it misses the
**spreading**: the heat enters on 30 mm and leaves over 100 mm, so under the
module the base is hotter than the mean. The two-dimensional solution is what
says by how much.

**Scaled units.** $X = x/W$, $Z = z/t_b$, and the rise above the air over
$\Theta = q_{\mathrm{mean}}/h_{\mathrm{eff}}$, the hand calculation's own
number, so that $\theta = 1$ means "what the hand calculation says". The base
plate becomes the unit square and

$$\varepsilon^2\,\theta_{XX} + \theta_{ZZ} = 0, \qquad
\theta_Z = \mathrm{Bi}\,g(X)\ \text{at}\ Z = 1, \qquad
\theta_Z = \mathrm{Bi}\,\theta\ \text{at}\ Z = 0, \qquad
\theta_X = 0\ \text{at}\ X = 0, 1$$

with $\varepsilon = t_b/W$ = 0.1 the aspect ratio, $\mathrm{Bi} =
h_{\mathrm{eff}} t_b/k$ the Biot number of the base plate, and
$g = q''/q_{\mathrm{mean}}$ = 3.33 on the footprint and 0 elsewhere.

**The exact solution.** Insulated ends make cosines in $x$ exact, and each
cosine has its own profile through the thickness:

$$\theta(x, z) = \sum_{n \ge 0} \cos\!\left(\frac{n\pi x}{W}\right) Z_n(z),
\qquad Z_0 \text{ linear}, \qquad Z_n = \frac{q_n}{k\lambda_n}\,
\frac{\cosh\lambda_n z + r_n \sinh\lambda_n z}{\sinh\lambda_n t_b + r_n\cosh\lambda_n t_b}$$

with $\lambda_n = n\pi/W$, $r_n = h_{\mathrm{eff}}/(k\lambda_n)$ and $q_n$
the cosine coefficients of the module's flux. Elementary functions, no
eigenvalue to find. Where the flux steps, at the footprint's edges, the series
converges slowly, so `hs.exact` sums that part in closed form (a dilogarithm)
and keeps only a remainder that dies exponentially: the field is exact to
rounding and costs milliseconds. **It is the truth** that both methods below
are scored against. The cell prints the data, draws the heat sink, and plots
the exact field.

In [ ]:
hs.describe_problem()
fig, ax = plt.subplots(figsize=(9.5, 4.6))
hs.draw_heatsink(ax)                                         # the cross-section, with the domain that is solved
plt.tight_layout(); plt.show()

xg, zg = np.linspace(0, hs.W_BASE, 401), np.linspace(0, hs.T_BASE, 41)
XG, ZG = np.meshgrid(xg, zg)
T_exact = hs.T_AIR + hs.exact(XG, ZG)                        # the exact temperature on a fine grid, °C
hot = hs.hottest(); hand = hs.THETA + hs.Q_MEAN * hs.T_BASE / hs.K_AL
print(f"hottest point: {hs.T_AIR + hot:.2f} °C, under the module's centre; the hand calculation says {hs.T_AIR + hand:.2f} °C, "
      f"so the spreading adds {hot - hand:.2f} K")
print(f"energy balance of the exact field: {hs.H_EFF * np.trapezoid(hs.exact(xg, np.zeros_like(xg)), xg) * hs.DEPTH:.2f} W leave through the fins of {hs.POWER:.0f} W")
fig, ax = plt.subplots(figsize=(9.5, 2.6))
c = ax.pcolormesh(XG * 1e3, ZG * 1e3, T_exact, cmap="inferno", shading="auto")
ax.plot([hs.MODULE[0] * 1e3, hs.MODULE[1] * 1e3], [hs.T_BASE * 1e3] * 2, color="tab:red", lw=4, label="the module")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("z  [mm]"); ax.legend(frameon=False, loc="lower right")
plt.colorbar(c, ax=ax, label="exact temperature  [°C]"); plt.title("the base plate: the exact field", fontsize=10)
plt.tight_layout(); plt.show()

**What you should see.** The fin efficiency 0.884 and $h_{\mathrm{eff}}$ =
322 W/(m²K), eight times the bare $h$: that is what the fins are for. The hand
calculation puts the module side at 87.47 °C; the exact solution puts the
hottest point at 92.45 °C, so the spreading adds 4.98 K, about a tenth of the
rise. The exact field sends 150.00 W through the fins. In the picture the
base is nearly uniform through its thickness and warmer under the module: the
heat spreads sideways through 10 mm of aluminium before it finds a fin.

---

## 4 · Finite differences, scored against the exact solution

**The grid fits the geometry.** The base plate is a rectangle, so a grid of
$n_x \times n_z$ nodes lays its nodes on every wall and there is no
staircase. The five-point stencil asks the equation at every interior node;
each wall is handled by a **ghost node** one spacing outside it, eliminated
with the wall's condition:

* insulated end: $\theta_{-1} = \theta_{1}$, a mirrored node;
* module face: $\theta_{N+1} = \theta_{N-1} + 2h_z\,\mathrm{Bi}\,g$, the flux
  enters through the ghost; a node on the footprint's edge carries half the
  flux;
* fin side: $\theta_{-1} = \theta_{1} - 2h_z\,\mathrm{Bi}\,\theta_0$, the
  convection.

One sparse solve gives every node (`hs.fdm_steady`). **The agreed accuracy is
0.01 K**, a tenth of what a thermocouple resolves; the exact solution lets us
ask for that. The cell refines the grid until it is met, and times one solve.

In [ ]:
TARGET = 0.01                                                 # K: the agreed accuracy
XG_s, ZG_s = XG / hs.W_BASE, ZG / hs.T_BASE                   # the scoring grid, scaled
rows_fdm = []
for nx, nz in ((21, 5), (41, 9), (81, 17), (161, 33), (321, 65)):
    X, Z, th = hs.fdm_steady(nx, nz)
    XX, ZZ = np.meshgrid(X * hs.W_BASE, Z * hs.T_BASE)
    err = np.abs(hs.kelvin(th) - hs.exact(XX, ZZ))           # against the exact field, on the grid's own nodes
    sec = online(lambda: hs.fdm_steady(nx, nz), reps=3)
    rows_fdm.append((nx, nz, err.max(), sec))
    i = np.unravel_index(err.argmax(), err.shape)
    print(f"  {nx:3d} x {nz:3d} nodes: worst error {err.max():.2e} K at x = {X[i[1]] * 100:.1f} mm on the {'module' if Z[i[0]] > 0.5 else 'fin'} side, "
          f"one solve {sec * 1e3:.1f} ms" + ("   <- meets 0.01 K" if err.max() <= TARGET else ""))
NX, NZ = 81, 17                                               # the coarsest grid that meets it

**What you should see.** The worst error halving with every halving of the
spacing - 3.73e-02, 1.85e-02, 9.16e-03, 4.54e-03, 2.26e-03 K - and always at
the footprint's edge on the module face, where the flux steps from 50 kW/m² to
nothing and the field has a corner in its gradient. Away from that edge
FDM is second order; at it, first. **81 × 17 nodes meet 0.01 K in a few
milliseconds.** A rectangle is the grid's home ground: every wall lies on a grid
line, so there is no staircase.

**Two ways to cover the base plate.** The cell below draws the grid that meets
the target and the points the PINN of section 5 trains on.

In [ ]:
X, Z = np.linspace(0, 1, NX), np.linspace(0, 1, NZ)
pts = hs.sample_base(2000)                                    # the PINN's collocation points
top, bottom, left, right = hs.sample_faces(200, 60)          # and its points on the four walls
fig, ax = plt.subplots(2, 1, figsize=(11, 5.2))
for xv in X: ax[0].plot([xv * 100, xv * 100], [0, 10], lw=0.3, color="0.5")
for zv in Z: ax[0].plot([0, 100], [zv * 10, zv * 10], lw=0.3, color="0.5")
ax[0].set_title(f"finite differences: {NX} x {NZ} nodes, every wall on a grid line", fontsize=10)
ax[1].plot(pts[:, 0] * 100, pts[:, 1] * 10, ".", ms=2, color="0.45", label="2000 points: the equation")
ax[1].plot(top[:, 0] * 100, top[:, 1] * 10, ".", ms=4, color="tab:red", label="the module face: the flux, denser at the footprint's edges")
ax[1].plot(bottom[:, 0] * 100, bottom[:, 1] * 10, ".", ms=4, color="tab:blue", label="the fin side: convection")
ax[1].set_title("PINN: scattered points, no mesh; the insulated ends need no points (section 5)", fontsize=10)
for q in ax:
    q.plot([hs.MODULE[0] * 100, hs.MODULE[1] * 100], [10.3, 10.3], color="tab:red", lw=3)
    q.set_aspect("equal"); q.set_xlim(-1, 101); q.set_ylim(-0.6, 11); q.set_ylabel("z  [mm]")
ax[1].set_xlabel("x  [mm]"); ax[1].legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3)
plt.tight_layout(); plt.show()

**What you should see.** Above, the grid: 1377 nodes, the footprint's edges
on grid lines. Below, the PINN's points: 2000 scattered in the plate, a
line on each face where a condition is asked, denser on the module face where
the flux steps, and nothing on the two ends.

---

## 5 · A PINN, and the comparison

**Build in what you know; let the network learn what you do not.** The hand
calculation is a solution of the equation and of two of the four walls (the
mean flux, the convection); only the spreading is missing. So the trial
function is

$$\hat\theta(X, Z) = \underbrace{1 + \mathrm{Bi}\,Z}_{\text{the hand calculation}}
\;+\; S\,\mathcal{N}\big(\cos \pi X,\ Z\big)$$

The network $\mathcal{N}$, four layers of 32 tanh neurons, learns the
spreading; $S$ = 0.1 is its size, fixed, not trained, there because the
spreading is a tenth of the rise. Its first input is **$\cos \pi X$ rather
than $X$**: a function of $\cos \pi X$ is even about both ends of the base,
so $\partial\hat\theta/\partial X$ = 0 at $X$ = 0 and $X$ = 1 holds exactly,
for any weights. This is **hard enforcement** of the insulated ends, by a
change of input rather than by a multiplier, and the ends need no loss term.

**The two faces stay loss terms**, with a weight - **soft enforcement**,
because the flux and the convection are conditions on a derivative:
$(\hat\theta_Z/\mathrm{Bi} - g)^2$ on the module face and
$(\hat\theta_Z/\mathrm{Bi} - \hat\theta)^2$ on the fin side. Both are divided
by Bi = 0.019 so that they are of order one, like the residual, which is
divided by Bi for the same reason. **The energy balance is a third loss term**:
the mean of $\hat\theta$ along the fin side must be exactly 1, because the heat
leaving is $h_{\mathrm{eff}}$ times that mean and must equal the 150 W that
enter. With no wall holding a temperature, this is what pins the level; without
it the PINN drifts by a kelvin.

**TODO 1** is the trial function; **TODO 2** the residual of the scaled
equation, $(\varepsilon^2\hat\theta_{XX} + \hat\theta_{ZZ})/\mathrm{Bi}$.

In [ ]:
SCALE = 0.1                                                   # the size of the spreading over the hand calculation
class BasePlate(torch.nn.Module):                             # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: cos(pi X), Z in
    def forward(self, p):
        X, Z = p[:, 0:1], p[:, 1:2]
        features = torch.cat([torch.cos(np.pi * X), Z], 1)    # even about both ends: the insulated ends, exactly
        return 1.0 + hs.BI * Z + SCALE * self.net(features)   # the trial function: the hand calculation plus the spreading

In [ ]:
def residual(model, p):
    th = model(p)                                             # the PINN's scaled rise
    return (hs.EPS ** 2 * d2(th, p, 0) + d2(th, p, 1)) / hs.BI   # the scaled Laplace equation, of order one; d2 = autograd, twice

In [ ]:
W_F, W_E = 10.0, 10.0                                         # the weights of the two faces and of the energy balance
P_in = to_tensor(pts, requires_grad=True)                     # collocation points, differentiable
P_top, P_bot = to_tensor(top, requires_grad=True), to_tensor(bottom, requires_grad=True)   # the two faces, differentiable
g_top = to_tensor(hs.footprint(top[:, 0])[:, None])           # q''/q_mean at each point of the module face
P_bal = to_tensor(np.c_[np.linspace(0, 1, 101), np.zeros(101)])   # where the fin-side mean is taken

def face_loss(model):
    flux = grad(model(P_top), P_top)[:, 1:2] / hs.BI - g_top         # the module face: theta_Z = Bi g; grad = autograd
    conv = grad(model(P_bot), P_bot)[:, 1:2] / hs.BI - model(P_bot)  # the fin side: theta_Z = Bi theta
    return flux.pow(2).mean() + conv.pow(2).mean()

def balance_loss(model):
    return (model(P_bal).mean() - 1.0).pow(2)                 # the heat out equals the heat in: the fin-side mean is 1

model = BasePlate().to(DEVICE)                                # four layers of 32
describe(model.net, n_collocation=len(P_in))                  # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, P_in).pow(2).mean() + W_F * face_loss(model) + W_E * balance_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
history = train_two_stage(model, loss_fn, adam_steps=1000, lbfgs_steps=400, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s (offline); residual {residual(model, P_in).pow(2).mean().item():.2e}, "
      f"faces {face_loss(model).item():.2e}, balance {balance_loss(model).item():.2e}")

In [ ]:
X, Z, th_fdm = hs.fdm_steady(NX, NZ)                          # the finite-difference answer on the grid that meets 0.01 K
XX, ZZ = np.meshgrid(X, Z)
T_true = hs.exact(XX * hs.W_BASE, ZZ * hs.T_BASE)             # the exact rise on the same nodes
Pn = to_tensor(np.c_[XX.ravel(), ZZ.ravel()])                 # the same nodes for the network
def pinn_field():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return hs.kelvin(to_numpy(model(Pn)).reshape(XX.shape))
T_f, T_p = hs.kelvin(th_fdm), pinn_field()
e_f, e_p = np.abs(T_f - T_true), np.abs(T_p - T_true)
t_fdm = online(lambda: hs.fdm_steady(NX, NZ), reps=3)         # one FDM solve, on this runtime
t_pin = online(pinn_field)                                    # one evaluation of the trained PINN on the same nodes, on this runtime
balance = hs.H_EFF * hs.THETA * np.trapezoid(to_numpy(model(P_bal)).ravel(), np.linspace(0, 1, 101)) * hs.W_BASE * hs.DEPTH

# the time column is what runs ONLINE: an FDM solve against an evaluation of the trained PINN on the same runtime.
# The training is offline, paid once, and reported on its own line.
print(f"{'':30s}{'unknowns':>10s}{'hottest':>11s}{'worst error':>14s}{'time online':>14s}")
print(f"  {'exact series':28s}{'-':>10s}{hs.T_AIR + T_true.max():8.2f} °C{'0 (the truth)':>14s}{'-':>14s}")
print(f"  {'finite differences, ' + str(NX) + ' x ' + str(NZ):28s}{NX * NZ:10d}{hs.T_AIR + T_f.max():8.2f} °C{e_f.max():11.2e} K{t_fdm * 1e3:11.1f} ms")
print(f"  {'PINN, 4 x 32':28s}{parameter_count(model):10d}{hs.T_AIR + T_p.max():8.2f} °C{e_p.max():11.2e} K{t_pin * 1e3:11.1f} ms   (trained offline in {train_time:.0f} s, once)")
print(f"energy balance of the PINN: {balance:.2f} W leave through the fins of {hs.POWER:.0f} W")
R1 = dict(hot_true=T_true.max(), hot_f=T_f.max(), hot_p=T_p.max(), e_f=e_f.max(), e_p=e_p.max(), t_fdm=t_fdm, t_pin=t_pin,
          train=train_time, n=parameter_count(model), grid=f"{NX} x {NZ}", nodes=NX * NZ, balance=balance)   # part 1's results, for the report

fig, axes = plt.subplots(3, 1, figsize=(10, 7.2))
for ax, vals, title, cmap in ((axes[0], hs.T_AIR + T_true, "exact  [°C]", "inferno"),
                              (axes[1], e_f, f"finite differences {NX} x {NZ}: error  [K]", "viridis"),
                              (axes[2], e_p, "PINN: error  [K]", "viridis")):
    c = ax.pcolormesh(XX * 100, ZZ * 10, vals, cmap=cmap, shading="auto")
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.set_ylabel("z  [mm]"); plt.colorbar(c, ax=ax, shrink=0.9)
axes[2].set_xlabel("x  [mm]")
plt.tight_layout(); plt.savefig(cc.output_path("Ex08_report_1.png"), dpi=150); plt.show()

**What you should see.** All three put the hottest point at 92.45 °C to
within a few hundredths. Finite differences are within 0.009 K of the exact
field everywhere, their error confined to the footprint's edges. The PINN
is within about 0.3 K: a smooth misfit of one to two tenths over most of the
plate, and its worst at the footprint's edges, where a smooth network cannot
follow the corner in the gradient. Its energy balance is 150 W to a tenth of
a watt, which the loss term made sure of. (The PINN's numbers vary a
little from run to run; FDM's do not.)

**By accuracy** the order is exact, FDM (0.009 K), PINN
(about 0.3 K). **By time online** both are milliseconds: one sparse solve of
1377 unknowns against one evaluation of 3300 weights at 1377 points, the
PINN a few times faster. Its training, four minutes on a CPU, is paid once
and offline.

---

## 6 · What part 1 says

* **On a rectangle, finite differences are hard to beat.** Every wall lies on
  a grid line, the stencil is second order away from the one corner in the
  gradient, and 0.01 K costs 1377 unknowns and a few milliseconds. The PINN
  reaches a thirtieth of that accuracy after minutes of training.
* **The exact solution is what made the comparison fair.** Both methods were
  scored against the truth, not against a third numerical answer; the
  footprint's edge shows up as the hard spot of both for the same reason.
* **What the PINN needed to get there is the lesson.** With four walls
  that carry a flux, a convection or nothing, and no wall holding a
  temperature, a plain PINN drifts in level by a kelvin. Building in the
  hand calculation, making the ends insulated by the choice of input, and
  asking for the energy balance as a loss term brought it to a few tenths.
  Each of those is physics the engineer already knew.
* **Online, the two are even.** Once trained, the PINN answers as fast as
  FDM solves. That matters when the answer is wanted many times - for a
  range of powers or footprints, or inside a controller - which is where the
  network earns its place, and in part 2, where one number of the model is
  unknown and has to be found from a thermocouple.
* **The fin-efficiency collapse is the one modelling step not checked here.**
  Mini project MP8.1A resolves the fins and finds how far the collapsed model
  is from the real cross-section.

---

# Part 2 · A pulse of power (L8.2)

The heat sink of Part 1 is at the air temperature when its module is switched
on at 150 W for twenty seconds - a motor started, a drive in overload - and
then off. It is the heat equation: the steady problem with one more term, the
heat the aluminium stores. **Nothing is loaded again**: `hs`, the libraries
and Part 1's results are still in memory, and Part 1's variable names are
reused below (its results are kept in `R1` for the report). The transient has
an exact solution too, so Crank–Nicolson and a PINN are scored against the
truth; then the same PINN finds the fins' coefficient from one thermocouple.

---

## 7 · The pulse

The heat sink of part 1 - base plate 100 × 10 mm, ten fins of 2 × 40 mm, forced
air at 40 °C - sits at the air temperature. At $t$ = 0 its module is switched
on at 150 W; at $t$ = 20 s it is switched off. A thermocouple is glued to the
module face of the base plate, 80 mm from the left end, 15 mm beside the
module. Wanted: the temperature of the base plate at every point and instant
over 150 s, the hottest point under the module through the pulse, and the
thermocouple's record.

## 8 · The data

| | |
|---|---|
| heat sink, module, air | as in part 1: $k$ = 167 W/(m·K), $h_{\mathrm{eff}}$ = 322 W/(m²K) on the fin side, 50 kW/m² on the footprint |
| storage | $\rho c_p$ = 2.43 MJ/(m³K); diffusivity $\alpha = k/\rho c_p$ = 6.87e-05 m²/s |
| the pulse | 150 W from 0 to 20 s, then nothing; the sink at 40 °C at $t$ = 0 |
| window | 0 to 150 s |
| thermocouple | on the module face at $x$ = 80 mm; a reading every 2 s, noise 0.05 K |

## 9 · The physics in time

**Storage is the new term** (L8.2's first section): the heat that arrives and
is not conducted away raises the temperature,

$$\rho c_p\,T_t = k\,(T_{xx} + T_{zz})$$

with the same three walls as before - the module's flux, now switched on and
off, the convection on the fin side, the insulated ends - and one more
condition, the field at $t$ = 0. The equation is **parabolic**: it has a
direction in time and needs one initial field.

**Three time scales**, and only two matter. Heat crosses the 10 mm thickness
in $t_b^2/\alpha$ = 1.5 s, so the base plate is uniform through its thickness
almost at once. It spreads along the 100 mm base in $W^2/(\pi^2\alpha)$ =
15 s: that is how long the hot spot under the module takes to form, and to
level out after the power goes off. And the whole plate warms and cools with
the **lumped time constant** $\rho c_p t_b / h_{\mathrm{eff}}$ = 75 s, the heat
it stores over the rate the fins remove it. A 20 s pulse is short against that:
the sink reaches only a quarter of its steady rise, and takes the rest of the
window to give it back.

**The hand calculation in time** is the lumped model: one temperature for the
whole base, rising as $1 - e^{-t/\tau}$ while the power is on and decaying
with the same $\tau$ after. It is exact for the mean of a plate this thin and
misses, as before, the spreading.

**Scaled units.** $X$, $Z$ and $\theta$ as in part 1, $\tau = t/t_{\mathrm{end}}$
with $t_{\mathrm{end}}$ = 150 s:

$$\theta_\tau = \mathrm{Fo}\,(\varepsilon^2\theta_{XX} + \theta_{ZZ}), \qquad
\mathrm{Fo} = \frac{\alpha\,t_{\mathrm{end}}}{t_b^2} = 103, \qquad
\theta_Z = \mathrm{Bi}\,g(X)\,s(\tau)\ \text{at}\ Z = 1, \qquad \theta = 0\ \text{at}\ \tau = 0$$

with $s(\tau)$ = 1 while the module is on and 0 after, and the fin side and
the ends as before. (In L8.2 $\tau$ is a time constant; in this notebook it is
only the scaled time.)

**The exact solution.** Each cosine of the steady series has its own history.
What has not yet arrived is expanded in the eigenfunctions of the convecting
face, $\cos\mu_j(t_b - z)$ with $\mu_j \tan(\mu_j t_b) = h_{\mathrm{eff}}/k$ -
the transcendental equation of Incropera ch. 5, solved to machine precision -
and every term dies as $e^{-\alpha(\lambda_n^2 + \mu_j^2)t}$:

$$\theta(x, z, t) = \theta_{\mathrm{steady}}(x, z) - \sum_{n, j} c_{nj}\cos\lambda_n x\,\cos\mu_j(t_b - z)\,e^{-\alpha(\lambda_n^2 + \mu_j^2)t}$$

A pulse is two steps, the second subtracted from the first after 20 s
(`hs.exact_transient`). It is exact to the number of terms kept, and for any
$t$ above a fraction of a second a hundred terms are far more than enough. The
cell prints the data, draws the heat sink with its thermocouple, and plots the
exact hottest point, the thermocouple's reading and the field at the end of the
pulse.

In [ ]:
hs.describe_problem()
print(f"  in time    : through the thickness t_b²/α = {hs.T_BASE ** 2 / hs.ALPHA:.1f} s, along the base W²/(π²α) = {hs.W_BASE ** 2 / (np.pi ** 2 * hs.ALPHA):.1f} s, "
      f"lumped time constant {hs.TAU_LUMPED:.1f} s; the pulse lasts {hs.T_ON:.0f} s, the window {hs.T_END:.0f} s; Fo = {hs.FO:.0f}")
fig, ax = plt.subplots(figsize=(9.5, 4.6))
hs.draw_heatsink(ax, show_sensor=True)
plt.tight_layout(); plt.show()

t_plot = np.arange(0.5, hs.T_END + 1e-9, 0.5)
centre = np.array([hs.exact_transient(np.array([hs.W_BASE / 2]), np.array([hs.T_BASE]), t)[0] for t in t_plot])   # the module's centre
sensor = np.array([hs.exact_transient(np.array([hs.SENSOR]), np.array([hs.T_BASE]), t)[0] for t in t_plot])        # the thermocouple, exact
lumped = hs.THETA * ((1 - np.exp(-t_plot / hs.TAU_LUMPED)) - np.where(t_plot > hs.T_ON, 1 - np.exp(-(t_plot - hs.T_ON) / hs.TAU_LUMPED), 0))
print(f"hottest point: {hs.T_AIR + centre.max():.2f} °C at {t_plot[centre.argmax()]:.1f} s; the lumped model's peak {hs.T_AIR + lumped.max():.2f} °C; "
      f"the thermocouple peaks at {hs.T_AIR + sensor.max():.2f} °C at {t_plot[sensor.argmax()]:.1f} s, after the power is off; "
      f"at {hs.T_END:.0f} s the sink is {centre[-1]:.2f} K above the air")
xg, zg = np.linspace(0, hs.W_BASE, 401), np.linspace(0, hs.T_BASE, 41); XG, ZG = np.meshgrid(xg, zg)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[1.1, 1]))
ax[0].plot(t_plot, hs.T_AIR + centre, lw=2, label="module centre, exact")
ax[0].plot(t_plot, hs.T_AIR + sensor, lw=2, label="thermocouple, exact")
ax[0].plot(t_plot, hs.T_AIR + lumped, "k--", lw=1, label="lumped model: one temperature")
ax[0].axvspan(0, hs.T_ON, color="tab:red", alpha=0.08); ax[0].text(hs.T_ON / 2, hs.T_AIR + 1, "power on", ha="center", fontsize=8, color="tab:red")
ax[0].set_xlabel("t  [s]"); ax[0].set_ylabel("temperature  [°C]"); ax[0].legend(frameon=False, fontsize=8)
c = ax[1].pcolormesh(XG * 1e3, ZG * 1e3, hs.T_AIR + hs.exact_transient(XG, ZG, hs.T_ON), cmap="inferno", shading="auto")
ax[1].set_aspect("equal"); ax[1].set_xlabel("x  [mm]"); ax[1].set_ylabel("z  [mm]"); ax[1].set_title("the base plate at 20 s, when the power goes off", fontsize=10)
plt.colorbar(c, ax=ax[1], label="°C"); plt.tight_layout(); plt.show()

**What you should see.** The module's centre climbing to 56.19 °C at 20 s, a
rise of 16.2 K where the lumped model says 10.8: the spreading is a third of
the rise while the power is on. When the power goes off the hot spot levels
out in about fifteen seconds and the whole plate then cools together, exactly
on the lumped curve. The thermocouple, 15 mm beside the module, **peaks four
seconds after the power is off**, at 50.0 °C: the heat under the module is
still arriving. At 150 s the sink is 2 K above the air.

---

## 10 · Finite differences in time, scored against the exact solution

**The same grid, stepped in time.** The five-point stencil and the ghost
nodes of part 1 give the space operator; **Crank–Nicolson** takes half of it
at the old time and half at the new, which is second order in time and stable
for any step. One factorisation of the matrix serves every step; the module's
flux is switched off at the step nearest 20 s (`hs.fdm_transient`).

**The agreed accuracy is 0.05 K**, the thermocouple's noise, at six recorded
instants: 7.5, 10, 20, 30, 60 and 150 s. The cell refines grid and step
together until it is met, scored against the exact field at every recorded
instant, and times one run.

In [ ]:
TARGET = 0.05                                                 # K: the agreed accuracy
REC_T = np.array([7.5, 10.0, 20.0, 30.0, 60.0, 150.0])        # the recorded instants, s
REC = REC_T / hs.T_END
exact_at = {}
for nx, nz, nt in ((41, 9, 150), (81, 17, 300), (161, 33, 600), (321, 65, 1200)):
    X, Z, TH = hs.fdm_transient(nx, nz, nt, REC)
    XX, ZZ = np.meshgrid(X * hs.W_BASE, Z * hs.T_BASE)
    errs = []
    for i, t in enumerate(REC_T):
        key = (nx, nz, t)
        exact_at[key] = hs.exact_transient(XX, ZZ, t)
        errs.append(np.abs(hs.kelvin(TH[i]) - exact_at[key]).max())
    sec = online(lambda: hs.fdm_transient(nx, nz, nt, REC), reps=3)
    print(f"  {nx:3d} x {nz:3d} nodes, {nt:5d} steps: worst error per instant", np.array2string(np.array(errs), precision=3),
          f"K, one run {sec:.2f} s" + ("   <- meets 0.05 K" if max(errs) <= TARGET else ""))
NX, NZ, NT = 321, 65, 1200                                    # the coarsest that meets it

**What you should see.** The worst error halving with every halving of the
spacing and the step - from about 0.4 K on the coarsest grid to 0.04 K on
321 × 65 nodes and 1200 steps - and largest just after the switch-on and the
switch-off, where the flux jumps and Crank–Nicolson is only first order. The
grid that meets 0.05 K has 20 865 unknowns and takes a few seconds for the
whole window: the transient costs FDM a thousand times the steady
problem, because it is solved once per step.

---

## 11 · A PINN in time, and the comparison

**Time is one more input** (L8.2's *Space–Time Sampling*): the PINN takes
$(X, Z, \tau)$ and the whole block of space and time is trained at once, with
half of the points in the first fifth of the window, where the field changes
fastest.

**Build in what the hand calculation knows.** The lumped model
$\theta_L(\tau)$ - the exponential rise and fall with $\tau_{\mathrm{lumped}}$
= 75 s - is a solution of the equation for the mean of the plate, and it is
zero at $\tau$ = 0. So the trial function is

$$\hat\theta(X, Z, \tau) = \theta_L(\tau) + S\,\tau\,\mathcal{N}\big(\cos\pi X,\ Z,\ \tau\big)$$

The factor $\tau$ makes the start exact for any weights - **the hard initial
condition** of L8.2, the first remedy for causality - and $\cos\pi X$ makes
the ends insulated, as in part 1. The network learns the spreading, which is
up to a third of the rise, so $S$ = 0.2. Its clock is $\sqrt{\tau}$ rather
than $\tau$: heat spreads as the square root of time, so the first seconds,
where the field changes fastest, are stretched out for the network. The two
faces stay loss terms, the module's flux now multiplied by $s(\tau)$, both
scaled by Bi; the residual is scaled by Bi as well. **The energy balance in
time is a third loss term**: at every instant the fin-side mean of the field
must equal the lumped model, because the lumped model is exactly the heat
balance of the whole plate. It holds the level of the field through the
window, as its steady form did in part 1.

**TODO 3** is the trial function; **TODO 4** the residual of the scaled heat
equation, $(\hat\theta_\tau/\mathrm{Fo} - \varepsilon^2\hat\theta_{XX} - \hat\theta_{ZZ})/\mathrm{Bi}$.

In [ ]:
SCALE = 0.2                                                   # the size of the spreading over the lumped rise
TAU_ON = hs.T_ON / hs.T_END                                   # the switch-off, in scaled time
RATE = hs.T_END / hs.TAU_LUMPED                               # t_end over the lumped time constant

def lumped(tau, rate=RATE):
    # the lumped base plate in units of THETA: rises as 1 - exp(-t/tau_L) while on, falls with the same tau_L after
    on = 1.0 - torch.exp(-rate * tau)
    off = torch.where(tau > TAU_ON, 1.0 - torch.exp(-rate * torch.clamp(tau - TAU_ON, min=0.0)), torch.zeros_like(tau))
    return on - off

class BaseInTime(torch.nn.Module):                            # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: cos(pi X), Z, sqrt(tau) in
    def rate(self):
        return RATE                                           # the lumped rate; the inverse problem of section 12 makes it trainable
    def forward(self, p):
        X, Z, tau = p[:, 0:1], p[:, 1:2], p[:, 2:3]
        features = torch.cat([torch.cos(np.pi * X), Z, torch.sqrt(tau + 1e-12)], 1)   # even about both ends; a square-root clock
        return lumped(tau, self.rate()) + SCALE * tau * self.net(features)   # the trial function: the lumped model, plus the spreading, zero at the start

In [ ]:
def residual(model, p):
    th = model(p)                                             # the PINN's scaled rise
    th_tau = grad(th, p)[:, 2:3]                              # the time derivative by autograd; column 2 is tau
    return (th_tau / hs.FO - hs.EPS ** 2 * d2(th, p, 0) - d2(th, p, 1)) / hs.BI   # the scaled heat equation, of order one

In [ ]:
W_F, W_E = 10.0, 10.0                                         # the weights of the two faces and of the energy balance
pts = hs.sample_base_time(4000)                               # (X, Z, tau), half of them in the first fifth of the window
top, bottom, left, right = hs.sample_faces_time(300, 60, 0)   # the faces, at random instants
P_in = to_tensor(pts, requires_grad=True)                     # collocation points, differentiable
P_top, P_bot = to_tensor(top, requires_grad=True), to_tensor(bottom, requires_grad=True)   # the two faces, differentiable
g_top = to_tensor((hs.footprint(top[:, 0]) * hs.pulse(top[:, 2]))[:, None])   # the module's flux, on while the power is on

tau_b, x_b = np.linspace(0, 1, 31), np.linspace(0, 1, 41)   # where the fin-side mean is taken: 41 points at 31 instants
P_bal = to_tensor(np.c_[np.tile(x_b, len(tau_b)), np.zeros(len(x_b) * len(tau_b)), np.repeat(tau_b, len(x_b))])   # (X, 0, tau) as a tensor
TAU_B = to_tensor(tau_b[:, None])                             # the same instants, for the lumped model

def face_loss(model, bi=hs.BI):
    flux = grad(model(P_top), P_top)[:, 1:2] / hs.BI - g_top             # the module face: theta_Z = Bi g s(tau)
    conv = grad(model(P_bot), P_bot)[:, 1:2] / hs.BI - (bi / hs.BI) * model(P_bot)   # the fin side: theta_Z = Bi theta
    return flux.pow(2).mean() + conv.pow(2).mean()

def balance_loss(model):
    mean_fin_side = model(P_bal).reshape(len(tau_b), len(x_b)).mean(1, keepdim=True)   # the fin-side mean at each instant
    return (mean_fin_side - lumped(TAU_B, model.rate())).pow(2).mean()   # equals the lumped model: the heat balance of the plate

model = BaseInTime().to(DEVICE)                               # four layers of 32
describe(model.net, n_collocation=len(P_in))                  # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, P_in).pow(2).mean() + W_F * face_loss(model) + W_E * balance_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
history = train_two_stage(model, loss_fn, adam_steps=1000, lbfgs_steps=300, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s (offline); residual {residual(model, P_in).pow(2).mean().item():.2e}, "
      f"faces {face_loss(model).item():.2e}, balance {balance_loss(model).item():.2e}")

In [ ]:
X, Z, TH = hs.fdm_transient(NX, NZ, NT, REC)                  # Crank-Nicolson on the grid that meets 0.05 K
XX, ZZ = np.meshgrid(X, Z)
T_true = np.array([exact_at[(NX, NZ, t)] for t in REC_T])     # the exact rise on the same nodes, at the recorded instants
def pinn_fields():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return np.array([hs.kelvin(to_numpy(model(to_tensor(np.c_[XX.ravel(), ZZ.ravel(), np.full(XX.size, r)]))).reshape(XX.shape)) for r in REC])   # the network on the grid at each instant
T_f, T_p = hs.kelvin(TH), pinn_fields()
e_f = np.abs(T_f - T_true).reshape(len(REC), -1).max(axis=1)  # worst error at each instant
e_p = np.abs(T_p - T_true).reshape(len(REC), -1).max(axis=1)
t_fdm = online(lambda: hs.fdm_transient(NX, NZ, NT, REC), reps=3)   # one run over the window, on this runtime
t_pin = online(pinn_fields)                                   # the trained PINN at the six instants on the same nodes, on this runtime
i20 = list(REC_T).index(20.0)

# the time column is what runs ONLINE: an FDM run over the window against an evaluation of the trained PINN at the
# recorded instants, on the same runtime. The training is offline, paid once, and reported on its own line.
print(f"{'':34s}{'unknowns':>10s}{'hottest at 20 s':>17s}{'worst error, any instant':>26s}{'time online':>14s}")
print(f"  {'exact series':32s}{'-':>10s}{hs.T_AIR + T_true[i20].max():14.2f} °C{'0 (the truth)':>26s}{'-':>14s}")
print(f"  {'Crank-Nicolson, ' + str(NX) + ' x ' + str(NZ) + ' x ' + str(NT):32s}{NX * NZ:10d}{hs.T_AIR + T_f[i20].max():14.2f} °C{e_f.max():23.2e} K{t_fdm:12.2f} s")
print(f"  {'PINN, 4 x 32':32s}{parameter_count(model):10d}{hs.T_AIR + T_p[i20].max():14.2f} °C{e_p.max():23.2e} K{t_pin:12.3f} s   (trained offline in {train_time:.0f} s, once)")
print("worst error per instant (7.5, 10, 20, 30, 60, 150 s):  FDM", np.array2string(e_f, precision=3), " PINN", np.array2string(e_p, precision=3))
R2 = dict(hot_true=T_true[i20].max(), hot_f=T_f[i20].max(), hot_p=T_p[i20].max(), e_f=e_f.max(), e_p=e_p.max(), t_fdm=t_fdm, t_pin=t_pin,
          train=train_time, n=parameter_count(model), grid=f"{NX} x {NZ} x {NT}", nodes=NX * NZ)   # part 2's results, for the report

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(t_plot, hs.T_AIR + centre, "k", lw=2, label="exact")
axes[0].plot(REC_T, hs.T_AIR + T_f.reshape(len(REC), -1).max(axis=1), "s", ms=7, label="Crank-Nicolson")
axes[0].plot(REC_T, hs.T_AIR + T_p.reshape(len(REC), -1).max(axis=1), "o", ms=5, label="PINN")
axes[0].set_xlabel("t  [s]"); axes[0].set_ylabel("hottest point  [°C]"); axes[0].legend(frameon=False)
axes[1].semilogy(REC_T, e_f, "s-", label=f"Crank-Nicolson, {NX} x {NZ} x {NT}")
axes[1].semilogy(REC_T, e_p, "o-", label="PINN")
axes[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 0.05 K")
axes[1].set_xlabel("t  [s]"); axes[1].set_ylabel("worst error over the plate  [K]"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(cc.output_path("Ex08_report_2.png"), dpi=150); plt.show()

**What you should see.** FDM is within 0.04 K of the exact field at
every recorded instant and puts the hottest point at 56.18 °C at 20 s, against
the exact 56.19. The PINN does not meet the agreed accuracy: it is within
0.06 K at 150 s and 0.1 K at 60 s, but off by one to two kelvin while the power
is on and at 20 s, where it puts the hottest point one to two kelvin low. Its error sits at
the footprint's edges in the first half-minute, where the spreading front is
a few millimetres wide and moves fast: features finer in space and time than
four layers of 32 follow in this training. After the switch-off the field
smooths and the PINN catches up. (Its numbers vary a little from run to
run; FDM's do not.)

**By accuracy** the order is exact, FDM, PINN, and only FDM meets
0.05 K throughout. **By time online** the PINN answers the six instants in
a few hundredths of a second where FDM needs three seconds for its 1200
steps - and the PINN could be read at any instant, not only the recorded
ones, without re-running anything. Its training is paid once, offline. What
would bring its early times in is L8.2's third remedy, **time windows**: a
PINN for the first thirty seconds alone, and another from there, which is
mini project MP8.2A's method.

---

## 12 · The inverse problem: the fins' coefficient from one thermocouple

The one number in this model that no handbook gives to better than twenty per
cent is $h_{\mathrm{eff}}$: it depends on the fan, the duct, the fin surface
and the fouling. Suppose it is **not known**, but the thermocouple recorded
the pulse, every 2 s with 0.05 K of noise. Can the same PINN find it?

It is L8.2's **Inverse Diffusivity** slide with a different unknown. One
number is trained beside the weights: $\mathrm{Bi} = \mathrm{Bi}_0 e^{\beta}$,
an exponential so that it stays positive. It is **not an output of the
network**: it is one more number the optimiser may change, and it enters the
loss in three places - the convection term on the fin side, the lumped
model built into the trial function, whose time constant is $\rho c_p
t_b/h_{\mathrm{eff}}$, and the energy balance that holds the field to it. The loss gains one term, the misfit to the readings.
$\beta$ starts at a coefficient 40 % too low. Nothing else changes.

**What identifies it.** While the power is on, the reading rises at a rate set
by the heat capacity, which is known; after the power is off, the reading
falls at a rate set by $h_{\mathrm{eff}}$ alone. The cooling tail is where the
information is, which is why the window runs to 150 s and not to 30. (Finding
the heat capacity as well, from the same record, is mini project MP8.2B.)

In [ ]:
t_s, r_s = hs.sensor_readings()                               # 75 readings: the rise above the air at the thermocouple, with noise
S_pts = to_tensor(np.c_[np.full_like(t_s, hs.SENSOR / hs.W_BASE), np.ones_like(t_s), t_s / hs.T_END])   # where and when they were read
S_val = to_tensor((r_s / hs.THETA)[:, None])                  # the readings, scaled

class Inverse(BaseInTime):                                    # the same network and trial function, and one more trainable number
    def __init__(self):
        super().__init__()                                    # a fresh network
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(0.6))))   # Bi = Bi_0 exp(beta), starts 40 % too low
    def bi(self):
        return hs.BI * torch.exp(self.beta)                   # the coefficient being identified, always positive
    def rate(self):
        return RATE * torch.exp(self.beta)                    # the lumped time constant moves with the unknown h_eff

inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 1000.0                                                  # the weight of the readings
def inverse_loss():
    data = (inv(S_pts) - S_val).pow(2).mean()                 # the misfit at the thermocouple
    return residual(inv, P_in).pow(2).mean() + W_F * face_loss(inv, inv.bi()) + W_E * balance_loss(inv) + W_D * data
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=1000, lbfgs_steps=300, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
h_found = float(inv.bi().detach()) / hs.BI * hs.H_EFF         # the identified number, back to W/(m²K)
print(f"h_eff: found {h_found:.1f} W/(m²K), true {hs.H_EFF:.1f} W/(m²K), off by {100 * (h_found / hs.H_EFF - 1):+.2f} %   ({inv_time:.0f} s)")
print(f"the lumped time constant that implies: {hs.RHO_CP * hs.T_BASE / h_found:.1f} s, true {hs.TAU_LUMPED:.1f} s")
R2["h_found"] = h_found                                       # the identified coefficient, for the report
with torch.no_grad():  # no gradient tracking: evaluation only
    fit = hs.kelvin(to_numpy(inv(S_pts)).ravel())
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(t_s, hs.T_AIR + r_s, ".", ms=4, label="thermocouple readings")
ax.plot(t_s, hs.T_AIR + fit, lw=1.5, label=f"the PINN's fit, h_eff = {h_found:.0f} W/(m²K)")
ax.axvspan(0, hs.T_ON, color="tab:red", alpha=0.08); ax.set_xlabel("t  [s]"); ax.set_ylabel("°C"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** The coefficient found about 5 % low - 306 against
the true 322 W/(m²K) - from one thermocouple and a start that was 40 % off,
in some minutes on a CPU; the fit runs through the readings to their noise.
The time constant that goes with it, 79 s against 75, is what the readings
really measured. Five per cent is within what any handbook gives for
$h_{\mathrm{eff}}$, and the error has the same origin as the forward
PINN's: the first half-minute, where the PINN's field under the module
is low, is also where the thermocouple's rise is fitted.

---

## 13 · What part 2 says

* **FDM meets the agreed 0.05 K throughout; the PINN only after the
  pulse.** On what an engineer wants - the module reaches 56 °C at the end of
  a 20 s pulse, 5 K above what the lumped model says, and the sink has given
  most of it back after two time constants - FDM is exact to a hundredth
  and the PINN one to two kelvin low at the peak.
* **The spreading is a transient of its own.** The hot spot forms and
  disappears in about fifteen seconds, the plate cools in seventy-five. A
  thermocouple beside the module peaks after the power is off.
* **The exact solution kept the comparison honest.** Both methods were scored
  against the truth at six instants, and both are worst where the flux jumps.
* **Online, the PINN is the faster of the two**, because a time-stepping
  FDM pays for every step and the PINN pays for none. It would be the one
  place in this notebook where the network wins the forward problem alone,
  if it met the accuracy; in this training it does not in the first
  half-minute, and the honest reading is that FDM wins the forward
  transient.
* **The inverse problem is what the PINN is for.** The same code, with one
  more trainable number and one more loss term, returned $h_{\mathrm{eff}}$
  from one surface thermocouple. FDM would have to be run again
  and again inside an optimiser; the PINN carries the unknown in its loss
  and in the lumped model built into it.

---

## 14 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex08_report.md` with the results of sections 5, 11 and 12 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L8.1 and L8.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which of the three mechanisms of L8.1 appear in this heat sink, where does each act, and which one is the field equation that was solved? (-> L8.1 Q1)": """
""",
    "No wall of the base plate holds a temperature. What fixes the level of the solution, how did the hand calculation use that, and what did the exact solution add to it? (-> L8.1 Q2)": """
""",
    "What did the trial function build in, and why does a network that takes cos(pi X) as its input satisfy the insulated ends for any weights? (-> L8.1 Q3)": """
""",
    "The two faces and the energy balance are loss terms with weights. What happened to the level of the field without the energy balance, and what did its weight trade? (-> L8.1 Q4)": """
""",
    "What does the storage term add to the steady problem of part 1, and what did the three time scales - through the thickness, along the base, the lumped time constant - tell you before anything was solved? (-> L8.2 Q1)": """
""",
    "The trial function is exactly zero at the start and carries the lumped model. Which of the lecture's remedies for causality is that, and where in time was the PINN's error largest? (-> L8.2 Q2)": """
""",
    "How was the transient checked, what made Crank-Nicolson first order at two instants, and why does the thermocouple peak after the power is off? (-> L8.2 Q3)": """
""",
    "In the inverse problem, what was trained besides the weights, in which places does it enter the loss, and which part of the record identified it? (-> L8.2 Q4)": """
""",
    "To conclude: for the steady sink solved once, for the pulse, and for the unknown coefficient, which method would you choose each time, and where did the PINN do something FDM could not do directly? (-> L8.1 and L8.2, Exercise slides)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| part | method | unknowns | hottest point | worst error against the exact solution | time online |", "|---|---|---|---|---|---|",
            f"| 1, steady | exact series | - | {hs.T_AIR + R1['hot_true']:.2f} °C | 0 (the truth) | - |",
            f"| 1, steady | FDM, {R1['grid']} | {R1['nodes']} | {hs.T_AIR + R1['hot_f']:.2f} °C | {R1['e_f']:.2e} K | {R1['t_fdm'] * 1e3:.1f} ms |",
            f"| 1, steady | PINN, 4 x 32 | {R1['n']} | {hs.T_AIR + R1['hot_p']:.2f} °C | {R1['e_p']:.2e} K | {R1['t_pin'] * 1e3:.1f} ms (trained offline in {R1['train']:.0f} s) |",
            f"| 2, at 20 s | exact series | - | {hs.T_AIR + R2['hot_true']:.2f} °C | 0 (the truth) | - |",
            f"| 2, at 20 s | Crank-Nicolson, {R2['grid']} | {R2['nodes']} | {hs.T_AIR + R2['hot_f']:.2f} °C | {R2['e_f']:.2e} K | {R2['t_fdm']:.2f} s |",
            f"| 2, at 20 s | PINN, 4 x 32 | {R2['n']} | {hs.T_AIR + R2['hot_p']:.2f} °C | {R2['e_p']:.2e} K | {R2['t_pin']:.3f} s (trained offline in {R2['train']:.0f} s) |",
            "", f"Energy balance of the steady PINN: {R1['balance']:.2f} W of {hs.POWER:.0f} W leave through the fins.",
            f"Inverse problem: h_eff found {R2['h_found']:.1f} W/(m²K), true {hs.H_EFF:.1f} W/(m²K), off by {100 * (R2['h_found'] / hs.H_EFF - 1):+.2f} %."]
    out = ["# Ex_08 — Heatsink Cooling of a Power Module", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex08_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures of sections 5 and 11, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08_report.md into Ex08_report.pdf, with any figure
# saved as Ex08_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex08_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex08_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 15 · Mini project proposal

Four problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP8.1A · The fins resolved

**The problem.** The notebook collapsed ten fins into one coefficient on the fin side, by the
straight-fin efficiency. Resolve them: solve conduction in the whole
cross-section, base plate and fins, with $h$ = 40 W/(m²K) on every face in
the air, and find how far the collapsed model is from the real thing.

| | |
|---|---|
| section | the heat sink of this notebook: base 100 × 10 mm, ten fins 2 × 40 mm at 10 mm pitch |
| module | 150 W on 30 mm of the top face |
| air | 40 °C, $h$ = 40 W/(m²K) on the fin faces, the fin tips and the bare base between the fins |

**Why a plain PINN is not enough.** The domain is a comb. A single network over a bounding box would have to be
told where the metal is; a Robin condition on every one of the thirty fin faces is
thirty loss terms, or one term on points sampled over all of them by length.

**What you build.** A PINN for $T(x, z)$ over the comb, points sampled in the metal only, the
convection on every air face as one weighted loss term over points sampled by
length, the module's flux on the top, scaled so every term is of order one.

**The ground truth you get.** `MP8.1A_truth.npz`, from `tools/miniprojects/ex08_truth.py`: finite
volumes on square cells of 0.25 mm that the rectilinear geometry fits exactly,
28 800 cells, converged to a few hundredths of a kelvin.

**Worked example.** 

| cells | hottest point | fin-side mean |
|---|---|---|
| 1.00 mm | 51.64 K | 45.89 K |
| 0.50 mm | 51.67 K | 45.82 K |
| 0.25 mm | 51.68 K | 45.80 K |
| the notebook's collapsed model | 52.45 K | 46.57 K |

The collapsed model is 0.77 K warm at the hottest point, 1.5 % of the rise: conservative, because the fin efficiency smears the fins' heat over the whole pitch where the real base is cooler above a fin root.

**Requirements.**

* The hottest point within **0.1 K** of the ground truth.
* The temperature within **0.2 K** everywhere in the metal.
* The heat through the air faces within **1 %** of 150 W.

**What you hand in.** The error map over the comb, the temperature along one fin against the ground truth and against the straight-fin formula, and one paragraph on where the collapsed model is wrong and why.

### MP8.1B · Two modules on one sink

**The problem.** Split the 150 W between two modules of 15 mm each, placed symmetrically with a
gap between them. How far apart should they sit? Train one PINN for every
gap from 0 to 40 mm, then find by a gradient through it the gap at which the
hottest point is lowest - and say why the answer is not simply "as far as possible".

| | |
|---|---|
| base plate, fins, air | as in this notebook |
| modules | two of 75 W on 15 mm each, symmetric about the centre, gap $d$ from 0 to 40 mm |

**Why a plain PINN is not enough.** A third input, the gap, and a flux pattern that moves with it: the trial
function must carry the hand calculation for every gap, and the flux loss must
be sampled where the footprints are for each one.

**What you build.** A PINN for $\theta(X, Z, d)$ with the hand calculation built in, the two faces
and the energy balance as loss terms, trained once over the range of gaps; then
`torch.autograd` of the hottest point with respect to $d$.

**The ground truth you get.** `MP8.1B_truth.npz`, from `tools/miniprojects/ex08_truth.py`: the exact
series by superposition of the two footprints, for every gap.

**Worked example.** 

| gap | hottest point | where |
|---|---|---|
| 0 mm | 52.45 K | 50.0 mm |
| 10 mm | 50.75 K | 41.2 mm |
| 20 mm | 49.76 K | 34.8 mm |
| 30 mm | 49.28 K | 28.3 mm |

Separating the modules lowers the hottest point, but the gain shrinks as each module nears an insulated end.

**Requirements.**

* The hottest point within **0.1 K** of the exact value at every gap.
* The best gap within **2 mm** of the exact optimum.
* One training for the whole range, no retraining per gap.

**What you hand in.** The hottest point against the gap, PINN against exact; the gap found by the gradient; and a paragraph on the two effects that compete.

### MP8.2A · Thermal cycling

**The problem.** A drive does not pulse once. Switch the module on for 20 s and off for 40 s,
ten times. What swing of temperature does the module see once the cycles
repeat, and how many cycles until they do? That swing is what ages a power
module's solder.

| | |
|---|---|
| heat sink, module, air | as in this notebook |
| the train | 150 W for 20 s, nothing for 40 s, from $t$ = 0, ten cycles: 600 s |

**Why a plain PINN is not enough.** Ten pulses in one window: the field has ten kinks in time, and a network
trained over 600 s at once resolves none of them well. Train it over one period
with the field at the end of the previous period as its start, or give it the
cycle number as an input.

**What you build.** A PINN for $\theta(X, Z, \tau)$ with the lumped train built in (a sum of the
exponentials of this notebook) and the spreading learnt; the start of each period
taken from the end of the last.

**The ground truth you get.** `MP8.2A_truth.npz`, from `tools/miniprojects/ex08_truth.py`: the exact
series by superposition of twenty steps, the module's centre every second.

**Worked example.** 

| cycle | peak | trough | swing |
|---|---|---|---|
| 1 | 16.19 K | 2.88 K | 13.31 K |
| 2 | 21.15 K | 9.25 K | 11.90 K |
| 5 | 24.88 K | 11.58 K | 13.30 K |
| 10 | 25.26 K | 11.80 K | 13.45 K |

The cycles repeat after about five; the module then swings by 13.5 K every minute about a mean 18 K above the air.

**Requirements.**

* The peak and the trough of every cycle within **0.3 K**.
* The number of cycles to a repeating swing, to the cycle.
* The field at the end of the tenth pulse within **0.5 K** everywhere.

**What you hand in.** The module's centre over 600 s, PINN against exact; the swing per cycle; and a paragraph on how the period was handled.

### MP8.2B · Heat capacity and coefficient together

**The problem.** Section 12 found $h_{\mathrm{eff}}$ with $\rho c_p$ known. Suppose neither is known:
the sink is of an alloy whose heat capacity is uncertain, in a duct whose
coefficient is unknown. Find both from the same thermocouple record, and show
which part of the record fixes which.

| | |
|---|---|
| record | the thermocouple of this notebook: 75 readings every 2 s, noise 0.05 K |
| unknowns | $\rho c_p$ (true 2.43 MJ/(m³K)) and $h_{\mathrm{eff}}$ (true 322 W/(m²K)); start both 30 % off |

**Why a plain PINN is not enough.** Two numbers that enter the same time constant $\rho c_p t_b / h_{\mathrm{eff}}$ can
trade against each other: the cooling tail fixes only their ratio. It is the rise
while the power is on, set by $\rho c_p$ alone, that separates them, and a network
that fits the tail well and the rise badly has found neither.

**What you build.** The inverse PINN of section 12 with two trainable numbers, both as exponentials,
entering the residual (through Fo), the fin-side term and the lumped model; then the
sensitivity of the fit to each, from the rise and from the tail separately.

**The ground truth you get.** `MP8.2B_readings.npz` to fit and `MP8.2B_truth.npz` to score against,
from `tools/miniprojects/ex08_truth.py`: the record and the two true values.

**Worked example.** With both unknown, the rise over the first 20 s determines $\rho c_p$ to a few per cent on its own; with it fixed, the tail from 20 to 150 s determines $h_{\mathrm{eff}}$ as in section 12.

**Requirements.**

* Both numbers within **5 %** of the true values.
* A plot of the misfit against each unknown, with the other at its found value, showing the rise and the tail separately.
* One paragraph on which readings fix which number.

**What you hand in.** The two numbers found, the two misfit plots, and the paragraph.